# Notebook 4: Deep Learning Frameworks — TensorFlow, Keras, and PyTorch

## Overview
This notebook introduces the three main deep learning frameworks and builds a **simple neural network** for image classification.

| Framework | Creator | Style |
|---|---|---|
| **TensorFlow 2.x** | Google | Static + eager execution |
| **Keras** | (included in TF 2) | High-level API on top of TF |
| **PyTorch** | Meta (Facebook) | Dynamic computation graph |

## Dataset — Fashion-MNIST (Kaggle)
**Source:** https://www.kaggle.com/datasets/zalando-research/fashionmnist

Fashion-MNIST contains 70,000 grayscale 28×28 images across 10 clothing categories.

### How to Download
```bash
pip install tensorflow torch torchvision kaggle matplotlib
# Place kaggle.json at ~/.kaggle/kaggle.json
kaggle datasets download -d zalando-research/fashionmnist \
  -p data/fashion_mnist --unzip
```

The dataset is also built into `tensorflow.keras.datasets.fashion_mnist` (used automatically if Kaggle CSV is absent).

In [ ]:
# ─── Install (uncomment if needed) ─────────────────────────────────────────
# !pip install tensorflow torch torchvision kaggle matplotlib

## 1. Check Available Frameworks

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# ── TensorFlow / Keras ────────────────────────────────────────────────────
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers
    TF_AVAILABLE = True
    print(f'TensorFlow : {tf.__version__}')
    print(f'Keras      : {keras.__version__}')
    print(f'GPU (TF)   : {len(tf.config.list_physical_devices("GPU"))} device(s)')
except ImportError:
    TF_AVAILABLE = False
    print('TensorFlow not installed. Run: pip install tensorflow')

# ── PyTorch ───────────────────────────────────────────────────────────────
try:
    import torch
    import torch.nn as nn
    import torch.optim as optim
    from torch.utils.data import DataLoader, TensorDataset
    PT_AVAILABLE = True
    print(f'PyTorch    : {torch.__version__}')
    print(f'CUDA (PT)  : {torch.cuda.is_available()}')
except ImportError:
    PT_AVAILABLE = False
    print('PyTorch not installed. Run: pip install torch torchvision')

print(f'\nTF_AVAILABLE = {TF_AVAILABLE}, PT_AVAILABLE = {PT_AVAILABLE}')

## 2. Load the Fashion-MNIST Dataset

In [ ]:
CLASS_NAMES = ['T-shirt/top','Trouser','Pullover','Dress','Coat',
               'Sandal','Shirt','Sneaker','Bag','Ankle boot']

if TF_AVAILABLE:
    # Load via Keras built-in helper (downloads ~30 MB on first run)
    (X_train_raw, y_train_raw), (X_test_raw, y_test_raw) = \
        keras.datasets.fashion_mnist.load_data()
    print(f'Train images : {X_train_raw.shape}')
    print(f'Test images  : {X_test_raw.shape}')
else:
    # Offline fallback: generate synthetic 28x28 'images'
    print('Generating synthetic Fashion-MNIST-like data (TF/Keras not available)...')
    np.random.seed(42)
    n_train, n_test = 6000, 1000   # small synthetic set
    X_train_raw = (np.random.rand(n_train, 28, 28) * 255).astype('uint8')
    X_test_raw  = (np.random.rand(n_test,  28, 28) * 255).astype('uint8')
    y_train_raw = np.random.randint(0, 10, n_train)
    y_test_raw  = np.random.randint(0, 10, n_test)
    print(f'Synthetic Train: {X_train_raw.shape}, Test: {X_test_raw.shape}')
    print('Note: real results require actual Fashion-MNIST data.')


In [ ]:
# Visualise sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train_raw[i], cmap='gray')
    ax.set_title(CLASS_NAMES[y_train_raw[i]])
    ax.axis('off')
plt.suptitle('Fashion-MNIST Sample Images')
plt.tight_layout()
plt.show()

## 3. Preprocessing

In [ ]:
# Normalise pixel values to [0, 1]
X_train_norm = X_train_raw.astype('float32') / 255.0
X_test_norm  = X_test_raw.astype('float32')  / 255.0

# Flatten 28×28 → 784 for fully-connected networks
X_train_flat = X_train_norm.reshape(-1, 784)
X_test_flat  = X_test_norm.reshape(-1, 784)

# Reshape for CNNs: (N, 28, 28, 1)
X_train_cnn = X_train_norm[..., np.newaxis]
X_test_cnn  = X_test_norm[...,  np.newaxis]

print(f'Flat shape : {X_train_flat.shape}')
print(f'CNN shape  : {X_train_cnn.shape}')

---
## Part A — TensorFlow / Keras
### A1. Multi-Layer Perceptron (MLP) with Keras Sequential API

In [ ]:
if TF_AVAILABLE:
    tf.random.set_seed(42)

    keras_mlp = keras.Sequential([
        layers.Input(shape=(784,)),
        layers.Dense(256, activation='relu'),
        layers.Dropout(0.3),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(10, activation='softmax')     # 10 Fashion-MNIST classes
    ], name='keras_mlp')

    keras_mlp.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    keras_mlp.summary()
else:
    print('TensorFlow not available — skipping Keras MLP.')

In [ ]:
if TF_AVAILABLE:
    # Train for a small number of epochs (increase for better results)
    history_mlp = keras_mlp.fit(
        X_train_flat, y_train_raw,
        validation_split=0.1,
        epochs=10,
        batch_size=256,
        verbose=1
    )

    test_loss, test_acc = keras_mlp.evaluate(X_test_flat, y_test_raw, verbose=0)
    print(f'\nKeras MLP — Test Accuracy: {test_acc:.4f}')

    # Plot training curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history_mlp.history['accuracy'],     label='Train')
    axes[0].plot(history_mlp.history['val_accuracy'], label='Validation')
    axes[0].set_title('Keras MLP — Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

    axes[1].plot(history_mlp.history['loss'],     label='Train')
    axes[1].plot(history_mlp.history['val_loss'], label='Validation')
    axes[1].set_title('Keras MLP — Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('TensorFlow not available — skipping training.')

### A2. Convolutional Neural Network (CNN) with Keras Functional API

In [ ]:
if TF_AVAILABLE:
    inputs = keras.Input(shape=(28, 28, 1))
    x = layers.Conv2D(32, kernel_size=(3, 3), activation='relu', padding='same')(inputs)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Conv2D(64, kernel_size=(3, 3), activation='relu', padding='same')(x)
    x = layers.MaxPooling2D(pool_size=(2, 2))(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(10, activation='softmax')(x)

    keras_cnn = keras.Model(inputs, outputs, name='keras_cnn')
    keras_cnn.compile(
        optimizer='adam',
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    keras_cnn.summary()

    history_cnn = keras_cnn.fit(
        X_train_cnn, y_train_raw,
        validation_split=0.1,
        epochs=10,
        batch_size=256,
        verbose=1
    )
    cnn_loss, cnn_acc = keras_cnn.evaluate(X_test_cnn, y_test_raw, verbose=0)
    print(f'\nKeras CNN — Test Accuracy: {cnn_acc:.4f}')
else:
    print('TensorFlow not available — skipping Keras CNN.')

---
## Part B — PyTorch
### B1. MLP in PyTorch

In [ ]:
if PT_AVAILABLE:
    torch.manual_seed(42)
    DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f'Using device: {DEVICE}')

    # Convert NumPy arrays → PyTorch tensors
    X_tr_t = torch.tensor(X_train_flat, dtype=torch.float32)
    y_tr_t = torch.tensor(y_train_raw,  dtype=torch.long)
    X_te_t = torch.tensor(X_test_flat,  dtype=torch.float32)
    y_te_t = torch.tensor(y_test_raw,   dtype=torch.long)

    train_loader = DataLoader(TensorDataset(X_tr_t, y_tr_t), batch_size=256, shuffle=True)
    test_loader  = DataLoader(TensorDataset(X_te_t, y_te_t), batch_size=256, shuffle=False)

    class MLP(nn.Module):
        def __init__(self):
            super().__init__()
            self.network = nn.Sequential(
                nn.Linear(784, 256), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(256, 128), nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(128, 10)
            )

        def forward(self, x):
            return self.network(x)

    pt_model   = MLP().to(DEVICE)
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.Adam(pt_model.parameters(), lr=1e-3)

    print(pt_model)
    total_params = sum(p.numel() for p in pt_model.parameters())
    print(f'Total parameters: {total_params:,}')
else:
    print('PyTorch not available — skipping PyTorch MLP.')

In [ ]:
if PT_AVAILABLE:
    def train_epoch(model, loader, criterion, optimizer, device):
        model.train()
        total_loss, correct = 0.0, 0
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            preds = model(X_batch)
            loss  = criterion(preds, y_batch)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(y_batch)
            correct    += (preds.argmax(1) == y_batch).sum().item()
        return total_loss / len(loader.dataset), correct / len(loader.dataset)

    def evaluate(model, loader, criterion, device):
        model.eval()
        total_loss, correct = 0.0, 0
        with torch.no_grad():
            for X_batch, y_batch in loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                preds       = model(X_batch)
                total_loss += criterion(preds, y_batch).item() * len(y_batch)
                correct    += (preds.argmax(1) == y_batch).sum().item()
        return total_loss / len(loader.dataset), correct / len(loader.dataset)

    N_EPOCHS = 10
    train_losses, train_accs = [], []
    test_losses,  test_accs  = [], []

    for epoch in range(1, N_EPOCHS + 1):
        tr_loss, tr_acc = train_epoch(pt_model, train_loader, criterion, optimizer, DEVICE)
        te_loss, te_acc = evaluate(pt_model, test_loader, criterion, DEVICE)
        train_losses.append(tr_loss); train_accs.append(tr_acc)
        test_losses.append(te_loss);  test_accs.append(te_acc)
        print(f'Epoch {epoch:02d}/{N_EPOCHS} | '
              f'Train Loss: {tr_loss:.4f}, Train Acc: {tr_acc:.4f} | '
              f'Test Loss: {te_loss:.4f}, Test Acc: {te_acc:.4f}')

    print(f'\nPyTorch MLP — Final Test Accuracy: {test_accs[-1]:.4f}')

    # Plot curves
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    epochs_range = range(1, N_EPOCHS + 1)
    axes[0].plot(epochs_range, train_accs, label='Train')
    axes[0].plot(epochs_range, test_accs,  label='Test')
    axes[0].set_title('PyTorch MLP — Accuracy')
    axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy'); axes[0].legend()

    axes[1].plot(epochs_range, train_losses, label='Train')
    axes[1].plot(epochs_range, test_losses,  label='Test')
    axes[1].set_title('PyTorch MLP — Loss')
    axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss'); axes[1].legend()

    plt.tight_layout()
    plt.show()
else:
    print('PyTorch not available — skipping training.')

## Summary

| Framework | Model | Notes |
|---|---|---|
| Keras (TF) | MLP | High-level Sequential API |
| Keras (TF) | CNN | Functional API for complex architectures |
| PyTorch | MLP | Custom training loop with fine-grained control |

### Key Takeaways
* **Keras** (via TensorFlow) provides a high-level API for fast prototyping.
* **PyTorch** gives fine-grained control via dynamic computation graphs.
* **CNNs** outperform MLPs on image data by exploiting spatial structure.
* Both frameworks produce similar accuracy — choice depends on use-case and preference.